# Stronger Head Detector — YOLO11s Fine-Tuning (Colab A100)

Trains a **stronger** single-class head detector on the real labeled **RPEE-Heads** dataset using **YOLO11s** at high resolution, validates on the held-out test split, and packages the model for download.

**Recommended training (run by this notebook):**
```
yolo detect train data=configs/head_dataset.yaml model=yolo11s.pt epochs=80 imgsz=1280
```

- Outputs are saved under `models/fine_tuned/head_detector_s/`.
- Validation runs on the official **val** split and the official **held-out test** split.
- A report (precision, recall, mAP50, mAP50-95, FPS) is written to `docs/HEAD_MODEL_TRAINING_S_REPORT.md`.
- Nothing is claimed as trained until `best.pt` actually exists on disk.

**Before running:** `Runtime > Change runtime type > GPU` (A100 recommended). YOLO11s for 80 epochs at imgsz=1280 is a multi-hour run; keep the tab active.

In [ ]:
# 1. Confirm CUDA GPU (A100 recommended)
import torch

print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU found. Runtime > Change runtime type > GPU (A100) before training.')
print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi

## 2. Load the project (git clone or zip upload)

Set `GITHUB_REPO_URL` to clone, or leave it blank to upload a zip of the `crowd-analysis` project.

In [ ]:
# 2. Load project from GitHub URL or uploaded zip
from pathlib import Path
import os
import shutil
import subprocess

PROJECT_DIR = Path('/content/crowd-analysis')
GITHUB_REPO_URL = ''  # Optional: set to git-clone instead of uploading a zip

def has_base_project_files(path: Path) -> bool:
    return (
        (path / 'requirements.txt').exists()
        and (path / 'src').is_dir()
        and (path / 'configs').is_dir()
    )

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

if GITHUB_REPO_URL.strip():
    subprocess.run(['git', 'clone', GITHUB_REPO_URL.strip(), str(PROJECT_DIR)], check=True)
else:
    from google.colab import files
    print('Upload a zip of the crowd-analysis project directory now.')
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
    if not zip_names:
        raise RuntimeError('No .zip file uploaded.')
    upload_zip = Path('/content') / zip_names[0]
    extract_dir = Path('/content/project_upload')
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(str(upload_zip), str(extract_dir))
    candidates = []
    if has_base_project_files(extract_dir):
        candidates.append(extract_dir)
    candidates.extend([p for p in extract_dir.rglob('*') if p.is_dir() and has_base_project_files(p)])
    if not candidates:
        raise RuntimeError('Could not find project files (requirements.txt, src/, configs/) in the uploaded zip.')
    shutil.copytree(candidates[0], PROJECT_DIR)

if not has_base_project_files(PROJECT_DIR):
    raise RuntimeError(f'Project setup failed: {PROJECT_DIR}')

os.chdir(PROJECT_DIR)
print('Project root:', Path.cwd())

In [ ]:
# 3. Install dependencies
!python -m pip install --upgrade pip -q
!python -m pip install -r requirements.txt -q

import ultralytics
print('Ultralytics:', ultralytics.__version__)

## 4. Download and prepare RPEE-Heads (train / val / held-out test)

Downloads the real RPEE-Heads archive (~1.1 GB), then builds the YOLO layout keeping the official **train**, **val**, and **test** splits. Every label row is validated; degenerate boxes are skipped and counted.

In [ ]:
# 4. Download the real RPEE-Heads dataset (~1.1 GB)
from pathlib import Path
import subprocess

RAW_DIR = Path('data/head_datasets/raw/rpee_heads')
ZIP_PATH = RAW_DIR / 'rpee_heads_dataset.zip'
RPEE_URL = 'https://ped.fz-juelich.de/data/machine_learning/2024_11_Recognition_In_Field_Studies/data/2024rpee_heads_dataset.zip'

RAW_DIR.mkdir(parents=True, exist_ok=True)
has_raw_images = any(RAW_DIR.rglob('*.jpg')) or any(RAW_DIR.rglob('*.jpeg')) or any(RAW_DIR.rglob('*.png'))
if not has_raw_images:
    print('Downloading RPEE-Heads (~1.1 GB)...')
    subprocess.run(['curl', '-L', '--fail', '--retry', '3', '-o', str(ZIP_PATH), RPEE_URL], check=True)
    print('Extracting...')
    subprocess.run(['python', '-m', 'zipfile', '-e', str(ZIP_PATH), str(RAW_DIR)], check=True)
else:
    print('Raw RPEE-Heads images already present; skipping download.')

n_imgs = sum(1 for _ in RAW_DIR.rglob('*.jpg')) + sum(1 for _ in RAW_DIR.rglob('*.jpeg')) + sum(1 for _ in RAW_DIR.rglob('*.png'))
print('Raw image count:', n_imgs)
if n_imgs == 0:
    raise RuntimeError('No RPEE-Heads images found after download/extract.')

In [ ]:
# 5. Prepare YOLO train/val/test layout with strict label validation
#    The official RPEE-Heads test split is kept for held-out evaluation.
from pathlib import Path
import os
import shutil
import yaml

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
OUT_DIR = Path('data/head_datasets/yolo/rpee_heads')

def classify_split(path: Path):
    parts = [p.lower() for p in path.parts]
    if 'testing' in parts or 'test' in parts:
        return 'test'
    if 'validation' in parts or 'val' in parts or 'valid' in parts:
        return 'val'
    if 'training' in parts or 'train' in parts:
        return 'train'
    return None

def find_label(image: Path):
    direct = image.with_suffix('.txt')
    if direct.exists():
        return direct
    parts = list(image.parts)
    for i in range(len(parts) - 1, -1, -1):
        if parts[i].lower() in {'images', 'image', 'imgs', 'jpegimages'}:
            mirrored = list(parts)
            mirrored[i] = 'labels'
            candidate = Path(*mirrored).with_suffix('.txt')
            if candidate.exists():
                return candidate
    matches = list(RAW_DIR.rglob(image.with_suffix('.txt').name))
    return matches[0] if len(matches) == 1 else None

def clean_label(label_path: Path):
    kept, bad = [], []
    for line_no, raw in enumerate(label_path.read_text(errors='replace').splitlines(), 1):
        line = raw.strip()
        if not line:
            continue
        cols = line.split()
        if len(cols) != 5:
            bad.append((str(label_path), line_no, 'field_count'))
            continue
        try:
            x, y, w, h = [float(v) for v in cols[1:]]
        except ValueError:
            bad.append((str(label_path), line_no, 'numeric'))
            continue
        if not all(0 <= v <= 1 for v in (x, y, w, h)) or w <= 0 or h <= 0:
            bad.append((str(label_path), line_no, 'normalized_or_size'))
            continue
        kept.append(f'0 {x:.6f} {y:.6f} {w:.6f} {h:.6f}')
    return kept, bad

if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
SPLITS = ['train', 'val', 'test']
for split in SPLITS:
    (OUT_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (OUT_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

stats = {s: {'images': 0, 'boxes': 0} for s in SPLITS}
missing_labels, invalid_rows = [], []

images = sorted(p for p in RAW_DIR.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS)
if not images:
    raise RuntimeError('No real RPEE-Heads images found after download/extract.')

for image in images:
    split = classify_split(image.relative_to(RAW_DIR))
    if split not in SPLITS:
        continue
    label = find_label(image)
    if label is None:
        missing_labels.append(str(image))
        continue
    kept, bad = clean_label(label)
    invalid_rows.extend(bad)
    image_dest = OUT_DIR / 'images' / split / image.name
    rel = os.path.relpath(image.resolve(), image_dest.parent.resolve())
    if image_dest.exists() or image_dest.is_symlink():
        image_dest.unlink()
    os.symlink(rel, image_dest)
    (OUT_DIR / 'labels' / split / f'{image.stem}.txt').write_text('\n'.join(kept) + ('\n' if kept else ''))
    stats[split]['images'] += 1
    stats[split]['boxes'] += len(kept)

if missing_labels:
    raise RuntimeError(f'Missing labels for {len(missing_labels)} images. Example: {missing_labels[:3]}')
if stats['train']['boxes'] == 0 or stats['val']['boxes'] == 0:
    raise RuntimeError(f'No labeled boxes in train/val: {stats}')

dataset_yaml = {
    'path': str(OUT_DIR.resolve()),
    'train': 'images/train',
    'val': 'images/val',
    'names': {0: 'head'},
}
HAS_TEST = stats['test']['images'] > 0 and stats['test']['boxes'] > 0
if HAS_TEST:
    dataset_yaml['test'] = 'images/test'

Path('configs').mkdir(exist_ok=True)
Path('configs/head_dataset.yaml').write_text(yaml.safe_dump(dataset_yaml, sort_keys=False))

print('Prepared stats:', stats)
print('Invalid rows skipped:', len(invalid_rows))
print('Held-out test split available:', HAS_TEST)
print(Path('configs/head_dataset.yaml').read_text())

## 6. Fine-tune the stronger head detector (YOLO11s)

Equivalent CLI: `yolo detect train data=configs/head_dataset.yaml model=yolo11s.pt epochs=80 imgsz=1280`. Outputs land under `models/fine_tuned/head_detector_s/`.

In [ ]:
# 6. Train YOLO11s
from pathlib import Path
import shutil
from ultralytics import YOLO

BASE_MODEL = 'yolo11s.pt'
EPOCHS = 80
IMAGE_SIZE = 1280
BATCH_SIZE = 16            # A100-40GB friendly at imgsz=1280; set -1 for auto, lower to 8 on CUDA OOM
PROJECT_OUT = (Path.cwd() / 'models/fine_tuned').resolve()
RUN_NAME = 'head_detector_s'
RUN_DIR = PROJECT_OUT / RUN_NAME
BEST_MODEL = RUN_DIR / 'weights' / 'best.pt'
DATA_YAML = str((Path.cwd() / 'configs/head_dataset.yaml').resolve())

model = YOLO(BASE_MODEL)
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    project=str(PROJECT_OUT),
    name=RUN_NAME,
    device=0,
    workers=8,
    exist_ok=True,
)

# Normalize the run directory if Ultralytics nested it under runs/detect.
if not BEST_MODEL.exists():
    cands = sorted(Path.cwd().glob('**/' + RUN_NAME + '/weights/best.pt'))
    if not cands:
        raise RuntimeError('Training finished but best.pt was not found under ' + str(PROJECT_OUT))
    found_run = cands[0].parent.parent
    shutil.copytree(found_run, RUN_DIR, dirs_exist_ok=True)

assert BEST_MODEL.exists(), 'best.pt missing; do not proceed.'
print('Best model:', BEST_MODEL.resolve())

## 7. Validate on val + held-out test, and benchmark FPS

Computes precision, recall, mAP50, mAP50-95 on the official val split and the held-out test split, then times inference FPS on the GPU.

In [ ]:
# 7. Validate (val + held-out test) and measure FPS
import time
from pathlib import Path
import yaml
import torch
from ultralytics import YOLO

DATA_YAML = str((Path.cwd() / 'configs/head_dataset.yaml').resolve())
cfg = yaml.safe_load(Path('configs/head_dataset.yaml').read_text())
detector = YOLO(str(BEST_MODEL))

def summarize(metrics):
    box = metrics.box
    return {
        'precision': round(float(box.mp), 4),
        'recall': round(float(box.mr), 4),
        'mAP50': round(float(box.map50), 4),
        'mAP50-95': round(float(box.map), 4),
    }

val_metrics = detector.val(data=DATA_YAML, split='val', imgsz=IMAGE_SIZE, device=0, verbose=False)
VAL_SUMMARY = summarize(val_metrics)
print('VAL:', VAL_SUMMARY)

TEST_SUMMARY = None
if 'test' in cfg:
    test_metrics = detector.val(data=DATA_YAML, split='test', imgsz=IMAGE_SIZE, device=0, verbose=False)
    TEST_SUMMARY = summarize(test_metrics)
    print('TEST (held-out):', TEST_SUMMARY)
else:
    print('No held-out test split present; reporting val metrics only.')

# FPS benchmark on the held-out test images (falls back to val images).
bench_split = 'test' if 'test' in cfg else 'val'
bench_dir = Path(cfg['path']) / cfg[bench_split]
bench_imgs = sorted(str(p) for p in bench_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})[:100]
for p in bench_imgs[:5]:
    detector.predict(p, imgsz=IMAGE_SIZE, device=0, verbose=False)  # warmup
t0 = time.perf_counter()
for p in bench_imgs:
    detector.predict(p, imgsz=IMAGE_SIZE, device=0, verbose=False)
elapsed = time.perf_counter() - t0
FPS = round(len(bench_imgs) / elapsed, 2) if elapsed > 0 else 0.0
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
print(f'FPS ({len(bench_imgs)} imgs @ imgsz={IMAGE_SIZE} on {GPU_NAME}): {FPS}')

In [ ]:
# 8. Show training curves / results.csv
from pathlib import Path
import pandas as pd

results_csv = RUN_DIR / 'results.csv'
if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    display(df.tail())
else:
    print('No results.csv at', results_csv)

!find {RUN_DIR} -maxdepth 2 -type f | sort

## 9. Write the training report

Fills a report template with the actual precision, recall, mAP50, mAP50-95, and FPS produced above.

In [ ]:
# 9. Write docs/HEAD_MODEL_TRAINING_S_REPORT.md from actual outputs
from pathlib import Path

REPORT = Path('docs/HEAD_MODEL_TRAINING_S_REPORT.md')
REPORT.parent.mkdir(parents=True, exist_ok=True)

primary = TEST_SUMMARY if TEST_SUMMARY else VAL_SUMMARY
primary_split = 'held-out test' if TEST_SUMMARY else 'val'

def fmt(summary):
    if not summary:
        return '- not available'
    return (f'- precision: {summary["precision"]}\n'
            f'- recall: {summary["recall"]}\n'
            f'- mAP50: {summary["mAP50"]}\n'
            f'- mAP50-95: {summary["mAP50-95"]}')

report = f'''# Head Model Training Report (YOLO11s)

## Dataset Used
RPEE-Heads only. No backup dataset and no synthetic data were used.

## Training Configuration
- base model: {BASE_MODEL}
- epochs: {EPOCHS}
- image size: {IMAGE_SIZE}
- batch size: {BATCH_SIZE}
- device used: {GPU_NAME}
- dataset YAML: `configs/head_dataset.yaml`
- output path: `models/fine_tuned/{RUN_NAME}/`
- best weights: `models/fine_tuned/{RUN_NAME}/weights/best.pt`

## Training Result
- training completed: {'YES' if BEST_MODEL.exists() else 'NO'}
- metrics available: {'YES' if results_csv.exists() else 'NO'}

## Accuracy (headline — primary split: {primary_split})
{fmt(primary)}
- FPS ({GPU_NAME}, imgsz={IMAGE_SIZE}): {FPS}

### Validation split
{fmt(VAL_SUMMARY)}

### Held-out test split
{fmt(TEST_SUMMARY)}

## Notes / Limitations
- FPS is measured on {GPU_NAME} at imgsz={IMAGE_SIZE}; deployment FPS on other hardware will differ.
- RPEE-Heads is railway-platform-relevant but NOT Indian-railway-specific. These are proxy metrics, not a production-readiness claim.
- Domain validation on approved Indian railway CCTV footage is still required.

## Next Steps
- compare against the YOLO11n head detector (`models/fine_tuned/head_detector/`)
- validate on labeled Indian Railway CCTV frames
- tune confidence/IoU thresholds for the deployment camera
'''
REPORT.write_text(report, encoding='utf-8')
print(REPORT.read_text())

In [ ]:
# 10. Package downloadable artifacts
from pathlib import Path
import zipfile

ARTIFACT_ZIP = Path('/content/head_detector_s_colab_outputs.zip')
artifact_paths = [
    BEST_MODEL,
    RUN_DIR / 'weights' / 'last.pt',
    RUN_DIR / 'results.csv',
    RUN_DIR / 'args.yaml',
    REPORT,
]
with zipfile.ZipFile(ARTIFACT_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in artifact_paths:
        if Path(path).exists():
            zf.write(path, arcname=str(path))
print('Artifact zip:', ARTIFACT_ZIP)
!ls -lh {ARTIFACT_ZIP}

In [ ]:
# 11. Download best.pt and the artifact bundle
from google.colab import files

files.download(str(BEST_MODEL))
files.download(str(ARTIFACT_ZIP))
print('Also: File > Download > Download .ipynb to keep the executed notebook.')

## 12. Export `best.pt` back into this repo

After the two downloads finish, on your **local machine** (repo root `crowd-analysis/`):

```bash
# 1. Place the downloaded weights into the repo
mkdir -p models/fine_tuned/head_detector_s/weights
cp ~/Downloads/best.pt models/fine_tuned/head_detector_s/weights/best.pt

# 2. (Optional) unzip the full bundle to also restore results.csv, args.yaml, the report
mkdir -p artifacts/colab_s
cp ~/Downloads/head_detector_s_colab_outputs.zip artifacts/colab_s/
cd artifacts/colab_s && unzip -o head_detector_s_colab_outputs.zip && cd ../..

# 3. Verify the weights load and record the checksum
python3 -c "from ultralytics import YOLO; YOLO('models/fine_tuned/head_detector_s/weights/best.pt'); print('best.pt OK')"
shasum -a 256 models/fine_tuned/head_detector_s/weights/best.pt

# 4. Run the repo's evaluation against the stronger model
python3 scripts/evaluate_head_detector.py \
  --model models/fine_tuned/head_detector_s/weights/best.pt \
  --source data/input_videos/sample.mp4 \
  --output data/outputs/head_demo_s.mp4 \
  --imgsz 1280
```

Then commit `models/fine_tuned/head_detector_s/weights/best.pt` and `docs/HEAD_MODEL_TRAINING_S_REPORT.md` once you have reviewed the numbers.